In [39]:
import pandas as pd
import numpy as np

In [40]:
df=pd.read_csv(r"C:\Users\shreyaash mogaveera\loan-default-risk-predictor\data\application_train.csv")

In [41]:
#loading the selected columns from EDA
with open('../data/selected_columns.json') as f:
    selected_cols= json.load(f)

In [42]:
df=df[selected_cols].copy()
df.shape

(307511, 47)

In [43]:
df.head()

,AMT_REQ_CREDIT_BUREAU_QRT,REGION_RATING_CLIENT_W_CITY,AMT_REQ_CREDIT_BUREAU_DAY,NAME_TYPE_SUITE,AMT_INCOME_TOTAL,NAME_CONTRACT_TYPE,ORGANIZATION_TYPE,LIVE_CITY_NOT_WORK_CITY,CNT_FAM_MEMBERS,NAME_EDUCATION_TYPE,...,AMT_REQ_CREDIT_BUREAU_WEEK,REG_REGION_NOT_LIVE_REGION,DEF_60_CNT_SOCIAL_CIRCLE,AMT_CREDIT,REG_CITY_NOT_WORK_CITY,EXT_SOURCE_2,EXT_SOURCE_1,DAYS_ID_PUBLISH,FLAG_DOCUMENT_5,LIVE_REGION_NOT_WORK_REGION
0,0.0,2,0.0,Unaccompanied,202500.0,Cash loans,Business Entity Type 3,0,1.0,Secondary / secondary special,...,0.0,0,2.0,406597.5,0,0.262949,0.083037,-2120,0,0
1,0.0,1,0.0,Family,270000.0,Cash loans,School,0,2.0,Higher education,...,0.0,0,0.0,1293502.5,0,0.622246,0.311267,-291,0,0
2,0.0,2,0.0,Unaccompanied,67500.0,Revolving loans,Government,0,1.0,Secondary / secondary special,...,0.0,0,0.0,135000.0,0,0.555912,NaN,-2531,0,0
3,NaN,2,NaN,Unaccompanied,135000.0,Cash loans,Business Entity Type 3,0,2.0,Secondary / secondary special,...,NaN,0,0.0,312682.5,0,0.650442,NaN,-2437,0,0
4,0.0,2,0.0,Unaccompanied,121500.0,Cash loans,Religion,1,1.0,Secondary / secondary special,...,0.0,0,0.0,513000.0,1,0.322738,NaN,-3458,0,0


In [44]:
df.isnull().sum()

AMT_REQ_CREDIT_BUREAU_QRT       41519
REGION_RATING_CLIENT_W_CITY         0
AMT_REQ_CREDIT_BUREAU_DAY       41519
NAME_TYPE_SUITE                  1292
AMT_INCOME_TOTAL                    0
NAME_CONTRACT_TYPE                  0
ORGANIZATION_TYPE                   0
LIVE_CITY_NOT_WORK_CITY             0
CNT_FAM_MEMBERS                     2
NAME_EDUCATION_TYPE                 0
FLAG_OWN_CAR                        0
DAYS_BIRTH                          0
EXT_SOURCE_3                    60965
TARGET                              0
DAYS_REGISTRATION                   0
OCCUPATION_TYPE                 96391
REG_REGION_NOT_WORK_REGION          0
AMT_ANNUITY                        12
OBS_30_CNT_SOCIAL_CIRCLE         1021
CODE_GENDER                         0
NAME_HOUSING_TYPE                   0
DAYS_EMPLOYED                       0
REGION_RATING_CLIENT                0
CNT_CHILDREN                        0
AMT_REQ_CREDIT_BUREAU_YEAR      41519
NAME_FAMILY_STATUS                  0
FLAG_OWN_REA

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Data columns (total 47 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   AMT_REQ_CREDIT_BUREAU_QRT    265992 non-null  float64
 1   REGION_RATING_CLIENT_W_CITY  307511 non-null  int64  
 2   AMT_REQ_CREDIT_BUREAU_DAY    265992 non-null  float64
 3   NAME_TYPE_SUITE              306219 non-null  object 
 4   AMT_INCOME_TOTAL             307511 non-null  float64
 5   NAME_CONTRACT_TYPE           307511 non-null  object 
 6   ORGANIZATION_TYPE            307511 non-null  object 
 7   LIVE_CITY_NOT_WORK_CITY      307511 non-null  int64  
 8   CNT_FAM_MEMBERS              307509 non-null  float64
 9   NAME_EDUCATION_TYPE          307511 non-null  object 
 10  FLAG_OWN_CAR                 307511 non-null  object 
 11  DAYS_BIRTH                   307511 non-null  int64  
 12  EXT_SOURCE_3                 246546 non-null  float64
 13 

In [46]:
#Handling missing values
num_cols=df.select_dtypes(include='number').columns.tolist()
keep=['TARGET','EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
for i in keep:
    num_cols.remove(i)

In [47]:
cat_cols=df.select_dtypes(include='object').columns.tolist()

In [48]:
#fill numerical cols with median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

#fill categorical cols with mode
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [49]:
df.isnull().sum()

AMT_REQ_CREDIT_BUREAU_QRT           0
REGION_RATING_CLIENT_W_CITY         0
AMT_REQ_CREDIT_BUREAU_DAY           0
NAME_TYPE_SUITE                     0
AMT_INCOME_TOTAL                    0
NAME_CONTRACT_TYPE                  0
ORGANIZATION_TYPE                   0
LIVE_CITY_NOT_WORK_CITY             0
CNT_FAM_MEMBERS                     0
NAME_EDUCATION_TYPE                 0
FLAG_OWN_CAR                        0
DAYS_BIRTH                          0
EXT_SOURCE_3                    60965
TARGET                              0
DAYS_REGISTRATION                   0
OCCUPATION_TYPE                     0
REG_REGION_NOT_WORK_REGION          0
AMT_ANNUITY                         0
OBS_30_CNT_SOCIAL_CIRCLE            0
CODE_GENDER                         0
NAME_HOUSING_TYPE                   0
DAYS_EMPLOYED                       0
REGION_RATING_CLIENT                0
CNT_CHILDREN                        0
AMT_REQ_CREDIT_BUREAU_YEAR          0
NAME_FAMILY_STATUS                  0
FLAG_OWN_REA

In [50]:
# filling EXT_SOURCE columns with -1, it will help the tree based models
for col in ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']:
    if col in df.columns:
        df[col]=df[col].fillna(-1)

In [55]:
#fixing DAYS_EMPLOYED anamaly
df['DAYS_EMPLOYED']=df['DAYS_EMPLOYED'].replace(365243,0)